# Is missingness itself a signal, or just absence?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [11]:
import polars as pl
from IPython.display import Markdown, display

df_txn = pl.scan_csv("../../kaggle/raw/train_transaction.csv", infer_schema_length=10000, null_values=[""]).collect()
df_id = pl.scan_csv("../../kaggle/raw/train_identity.csv", infer_schema_length=10000, null_values=[""]).collect()
df = df_txn.join(df_id, on="TransactionID", how="left")

## Missingness as Structure

This is not a dataset with a few gaps. It is a dataset whose column families are defined
by when they are absent.



In [14]:
families = {
    "`V1-V339` (anonymized)": [c for c in df.columns if c.startswith("V")],
    "`id_01-id_38` (identity)": [c for c in df.columns if c.startswith("id_")],
    "`D1-D15` (time deltas)": [c for c in df.columns if c.startswith("D") and c != "DeviceType" and c != "DeviceInfo"],
    "`C1-C14` (counters)": [c for c in df.columns if c.startswith("C")],
    "`M1-M9` (match flags)": [c for c in df.columns if c.startswith("M")],
    "`card1-card6`": [c for c in df.columns if c.startswith("card")],
    "`addr` / `dist`": [c for c in df.columns if c.startswith(("addr", "dist"))],
    "Email domains": [c for c in df.columns if "emaildomain" in c]
}

data = []
for name, cols in families.items():
    if not cols:
        continue
    rates = [df.select(pl.col(c).null_count()).item() / df.height for c in cols]
    data.append({
        "Family": name,
        "Columns": len(cols),
        "Min null rate": f"{min(rates)*100:.1f}%",
        "Median": f"{pl.Series(rates).median()*100:.1f}%",
        "Max": f"{max(rates)*100:.1f}%"
    })

df_fam = pl.DataFrame(data)
display(Markdown(df_fam.to_pandas().to_markdown(index=False)))

| Family                   |   Columns | Min null rate   | Median   | Max   |
|:-------------------------|----------:|:----------------|:---------|:------|
| `V1-V339` (anonymized)   |       339 | 0.0%            | 47.3%    | 86.1% |
| `id_01-id_38` (identity) |        38 | 75.6%           | 82.4%    | 99.2% |
| `D1-D15` (time deltas)   |        15 | 0.2%            | 52.5%    | 93.4% |
| `C1-C14` (counters)      |        14 | 0.0%            | 0.0%     | 0.0%  |
| `M1-M9` (match flags)    |         9 | 28.7%           | 47.7%    | 59.3% |
| `card1-card6`            |         6 | 0.0%            | 0.3%     | 1.5%  |
| `addr` / `dist`          |         4 | 11.1%           | 35.4%    | 93.6% |
| Email domains            |         2 | 16.0%           | 46.4%    | 76.8% |

Two families stand out for opposite reasons. `C1-C14` are fully populated — no nulls
anywhere — which is why `C13` turns out to be the model's single strongest driver.
`card1` has zero nulls, which is why it was chosen as the card entity for the velocity
features and why those features need no null guard.

`null_count_V_block`, the count of missing V columns per row, ranges from **11 to 274**.
No row has a complete V-block and no row is missing all of it.

